# TD/TP 4 — LLMs in practice: tokens, retrieval, and grounded answers

**IMT · Introduction to Machine Learning · Google Colab**  
**Estimated duration:** 55–75 min (individual/group work + collective correction)  
**Based on course material:** Part 5: GPT/BERT context, tokenization, finite context windows; Part 6: document chunks, embeddings/retrieval, RAG, source-grounded answers

**Learning objectives:** reason about tokens; chunk technical reports with provenance; implement lexical retrieval and rank fusion; build a grounded prompt; optionally run a small pretrained language model.

### How to use this notebook
1. Read the questions, run the prepared cells in order, and complete every **✍️ TODO** cell yourself.
2. The data preparation and plotting helpers are supplied so you can focus on machine learning.
3. If you get stuck, discuss your approach with a classmate before consulting the instructor.
4. You will receive the complete solution notebook after the collective correction.

**Environment:** Google Colab → Runtime → Run all (for the solutions). A CPU is sufficient. The datasets are generated locally or bundled with `scikit-learn`; no paid API key is required.

## Scenario: ask questions across maintenance reports

A language model has a finite context window and can produce unsupported answers. **Retrieval-augmented generation (RAG)** addresses this by selecting relevant excerpts *before* asking the model to answer.

We'll create a miniature RAG application inspired by the final part of the course.

**Important:** The example reports below are **invented for teaching**; their measurements are not from real equipment. The mandatory notebook needs only NumPy and scikit-learn. An **optional pretrained language-model demo** at the end requires downloading a public model (no API key).

In [ ]:
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Synthetic, fictional engineering reports for teaching ONLY.
reports = [
    {"id":"R01", "title":"Inspection — gearbox A17 — Lyon", "text":
     "Site: Lyon. Asset: gearbox A17. Inspection date: 2025-01-14.\n\n"
     "Measurement: gear tooth wear rate was 0.18 mm/year. No cracked tooth observed.\n\n"
     "Action: continue routine monitoring every six months."},
    {"id":"R02", "title":"Inspection — gearbox B04 — Nantes", "text":
     "Site: Nantes. Asset: gearbox B04. Inspection date: 2025-02-02.\n\n"
     "Measurement: gear tooth wear rate was 0.53 mm/year. Visible pitting on the gear surface.\n\n"
     "Action: plan replacement of the damaged gear and inspect lubrication."},
    {"id":"R03", "title":"Inspection — gearbox C12 — Marseille", "text":
     "Site: Marseille. Asset: gearbox C12. Inspection date: 2025-03-17.\n\n"
     "Measurement: gear tooth wear rate was 0.31 mm/year. Vibration within acceptable limits.\n\n"
     "Action: inspect again in three months."},
    {"id":"R04", "title":"Inspection — gearbox D08 — Lille", "text":
     "Site: Lille. Asset: gearbox D08. Inspection date: 2025-04-05.\n\n"
     "Measurement: gear tooth wear rate was 0.47 mm/year. Small surface crack detected.\n\n"
     "Action: stop the machine and arrange a specialist safety assessment."},
    {"id":"R05", "title":"Inspection — pump P11 — Bordeaux", "text":
     "Site: Bordeaux. Asset: pump P11. Inspection date: 2025-04-22.\n\n"
     "Measurement: outlet pressure was 5.2 bar; temperature was 62 C. No leakage found.\n\n"
     "Action: keep the normal maintenance interval."},
    {"id":"R06", "title":"Inspection — bearing E06 — Paris", "text":
     "Site: Paris. Asset: bearing E06. Inspection date: 2025-05-03.\n\n"
     "Measurement: bearing vibration was 9.1 mm/s; abnormal heating observed.\n\n"
     "Action: inspect bearing alignment and renew lubrication."},
    {"id":"R07", "title":"Inspection — gearbox F22 — Toulouse", "text":
     "Site: Toulouse. Asset: gearbox F22. Inspection date: 2025-05-21.\n\n"
     "Measurement: gear tooth wear rate was 0.12 mm/year. Lubricant sample was clean.\n\n"
     "Action: no corrective work required."},
    {"id":"R08", "title":"Inspection — gearbox G19 — Rennes", "text":
     "Site: Rennes. Asset: gearbox G19. Inspection date: 2025-06-01.\n\n"
     "Measurement: gear tooth wear rate was 0.62 mm/year. Severe corrosion visible.\n\n"
     "Action: replace the affected component and investigate water ingress."},
]
print(f'{len(reports)} fictional source documents')
print(reports[1]['title'], '\n', reports[1]['text'])

## 1. ✍️ TODO — tokens and the context-window problem

A token is **not the same thing as a word**. Write a simple regex-based word/punctuation tokenizer `re.findall(r"\w+|[^\w\s]", text)` and compare the number of characters, whitespace-delimited words, and proxy tokens.

**Important:** This is a classroom **proxy**, not GPT/BERT tokenization. Real systems usually use trained subword/byte tokenizers (e.g., BPE).

In [ ]:
example = 'The wear rate is 0.53 mm/year — check lubrication!'

def simple_tokens(text):
    # TODO: return list of word/punctuation matches using re.findall.
    raise NotImplementedError('Implement a simple tokenizer')

print('Characters:', len(example))
print('Whitespace words:', len(example.split()))
print('Regex token proxy:', len(simple_tokens(example)), simple_tokens(example))

## 1b. ✍️ TODO — visualize GPT-style causal attention vs BERT-style attention

A key idea in Part 5 is the difference between **encoder (BERT-style, bidirectional)** and **decoder-only (GPT-style, causal)** attention. For simplicity, we use **fixed random vectors** as token features (no training here), and use them as both query and key vectors.

Compute the attention matrix using:

- Scores: `scores = E @ E.T / sqrt(d)` (provided)
- **Bidirectional attention**: row-wise softmax over every token position.
- **Causal attention**: apply a mask of `-1e9` to *future* positions (`column > row`) before softmax.

**Question:** Can the representation of the first token attend to the last token in each case? Why does this difference matter for next-token generation?


In [ ]:
tokens = ['The', 'gear', 'has', 'visible', 'wear']
E = np.random.default_rng(42).normal(size=(len(tokens), 4))  # toy token features
scores = E @ E.T / np.sqrt(E.shape[1])

def attention_weights(scores, causal=False):
    # TODO: if causal, replace strictly upper-triangular scores by -1e9.
    # TODO: apply a stable row-wise softmax; rows must sum to 1.
    raise NotImplementedError('Build causal and bidirectional attention matrices')

bidirectional = attention_weights(scores, causal=False)
causal = attention_weights(scores, causal=True)
assert np.allclose(causal.sum(axis=1), 1)
assert np.allclose(causal[np.triu_indices(len(tokens), 1)], 0)
fig, axs = plt.subplots(1,2,figsize=(10,4))
for ax, weights, title in zip(axs,[bidirectional, causal],['BERT-style: bidirectional','GPT-style: causal']):
    ax.imshow(weights, cmap='viridis', vmin=0, vmax=1)
    ax.set(title=title,xticks=range(len(tokens)),yticks=range(len(tokens)))
    ax.set_xticklabels(tokens,rotation=40,ha='right')
    ax.set_yticklabels(tokens)
    ax.set_xlabel('Key / attended-to position')
    ax.set_ylabel('Query / current position')
plt.tight_layout();plt.show()

## 2. ✍️ TODO — chunk documents, preserving their sources

A RAG system doesn't have to pass whole reports into the model. Split each report at blank lines (`"\n\n"`) so every paragraph becomes one **chunk**. Store at least `report_id`, `title`, `section`, and `text` with each chunk.

**Checkpoint:** each synthetic report has three paragraphs, so expect **24 chunks**.

In [ ]:
def make_chunks(reports):
    chunks = []
    for report in reports:
        # TODO: split report['text'] into paragraphs separated by blank lines.
        # TODO: append one dict per paragraph with keys: report_id, title, section, text.
        # Use section numbers 1, 2, 3.
        raise NotImplementedError('Chunk reports while preserving metadata')
    return chunks

chunks = make_chunks(reports)
assert len(chunks) == 24, f'Expected 24 chunks, got {len(chunks)}'
assert all('report_id' in c and 'text' in c for c in chunks)
print(pd.DataFrame(chunks).head(6).to_string(index=False))

## 3. ✍️ TODO — build a searchable index and retrieve evidence

Fit a `TfidfVectorizer(ngram_range=(1,2))` on chunk text. Encode a question using the **same fitted vectorizer** and rank chunks by cosine similarity.

This implements a lightweight **lexical** retriever, rather than the **neural multilingual-E5 embeddings** used in your course's RAG case study. Both produce numerical search representations, but lexical TF-IDF does **not** understand meaning like a trained embedding model.

Implement `retrieve(question, k=4)` returning a list of dicts (each with its source metadata and a `score`).

In [ ]:
# TODO: fit word_vectorizer and calculate the sparse index matrix `word_index`.
raise NotImplementedError('Vectorize report chunks')

def retrieve(question, k=4):
    # TODO: transform question and compute cosine similarity against `word_index`.
    # TODO: return k chunk dicts sorted by similarity, each with 'score'.
    raise NotImplementedError('Build lexical retrieval')

question = 'Which gearboxes have a high gear tooth wear rate?'
hits = retrieve(question, k=5)
for h in hits:
    print(h['report_id'], f"score={h['score']:.3f}", h['text'])

## 4. ✍️ TODO — combine two retrieval rankings (RRF)

Your course's Part 6 combines **semantic retrieval + BM25 + metadata** using **reciprocal rank fusion (RRF)**. We'll implement a small *analogue*: fuse **word TF-IDF** and **character n-gram TF-IDF** rankings. These are **two lexical retrieval methods**, not substitutes for neural E5 and BM25.

For a document ranked at position `r` (1-indexed), add `1/(60+r)` from each ranking. Higher combined scores win.

**Checkpoint:** return `k` chunks with a new `score` field. Note how rankings can change.

In [ ]:
char_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5))
char_index = char_vectorizer.fit_transform([c['text'] for c in chunks])

def retrieve_rrf(question, k=4, rrf_constant=60):
    # TODO: compute cosine similarities for the query in both TF-IDF indices.
    # TODO: convert each sorted ranking into its 1-indexed ranks.
    # TODO: fuse: 1/(rrf_constant + word_rank) + 1/(rrf_constant + char_rank).
    # TODO: return top-k chunk dicts with score.
    raise NotImplementedError('Implement reciprocal rank fusion')

for h in retrieve_rrf('corrosion of gearbox G19', k=3):
    print(h['report_id'], h['text'], f"score={h['score']:.4f}")

## 4b. ✍️ TODO — expand the retrieved chunk to its report context

A query can retrieve the right **report**, but only the *title or metadata paragraph*, omitting the **action** paragraph needed to answer. This happens in real RAG applications too.

As a simple remedy, keep the first **two distinct report IDs** returned by the retriever, then include **all three paragraphs from each report**. In a real large system, you would use parent-document retrieval or targeted neighboring chunks, with a strict context budget.

**Question:** Why can looking only at the top-ranked *individual* chunks be misleading?


In [ ]:
def expand_to_full_reports(hits, chunks, max_reports=2):
    # TODO: collect the first max_reports *distinct* report IDs from hits.
    # TODO: return all the corresponding chunks from the original `chunks` list.
    raise NotImplementedError('Expand retrieved chunks to their parent report')

expanded = expand_to_full_reports(retrieve_rrf('Action recommended for gearbox B04', k=6), chunks, max_reports=2)
print([(x['report_id'],x['section']) for x in expanded])

## 5. ✍️ TODO — construct a grounded LLM prompt

Pass the **question** and retrieved text to a model with source IDs. Tell the model to use **only retrieved evidence**, to cite each supporting report ID, and to say it cannot find the answer when evidence is missing.

**Why this is important:** RAG *reduces* unsupported answers but doesn't guarantee correctness; retrieved evidence may itself be wrong or incomplete.

In [ ]:
def build_prompt(question, hits):
    # TODO: include instructions requiring cited answers and an explicit unknown fallback.
    # Include each passage formatted as e.g. [R02 / section 2] <text>
    raise NotImplementedError('Build source-grounded prompt')

query = 'What action is recommended for gearbox B04?'
evidence = expand_to_full_reports(retrieve_rrf(query, k=6), chunks, max_reports=2)
prompt = build_prompt(query, evidence)
print(prompt)
assert 'R02' in prompt, 'Relevant report R02 should be included in this prompt'
assert "plan replacement" in prompt, "Retrieved context must include B04 recommendation"

## 6. Check retrieval, source grounding, and numerical reasoning

RAG retrieval is **not the same as exact numerical filtering**. For instance, the question "Which gearboxes have wear rate greater than 0.40 mm/year?" asks for **all matching reports**, not merely semantically similar chunks. We can verify such a claim with a simple exact extractor, and use the results to check an LLM answer.

Below is a **provided reference checker**, not an LLM. It extracts wear numbers from the fabricated reports and applies the threshold; it should return R02, R04, and R08.

In [ ]:
def verify_wear_above_threshold(reports, threshold=0.40):
    matches = []
    for report in reports:
        # This regex is specific to our synthetic report format, NOT a generic PDF parser.
        found = re.search(r'wear rate was (\d+\.\d+) mm/year', report['text'])
        if found and float(found.group(1)) > threshold:
            matches.append((report['id'], float(found.group(1))))
    return matches

verified = verify_wear_above_threshold(reports, 0.40)
print('Exact check:', verified)
assert {report_id for report_id, _ in verified} == {'R02','R04','R08'}
print('RAG question: Which gearboxes have wear above 0.40 mm/year?')
print('Top retrieved chunks:')
for h in retrieve_rrf('gear tooth wear mm/year above 0.40',k=6):
    print(h['report_id'], '|',h['text'])

## 7. OPTIONAL — run a real pretrained language model (internet required)

This cell downloads the small public pretrained **encoder–decoder** instruction model `google/flan-t5-small` from Hugging Face, unless already cached. It is a small demo model, **not a frontier-scale LLM** and **not GPT/BERT**; it illustrates generation after retrieval. It may answer imperfectly or omit citations.

**Compatibility note:** Transformers v5 removed the `text2text-generation` pipeline. This notebook uses `AutoTokenizer`, `AutoModelForSeq2SeqLM`, and `model.generate()` instead, compatible with Transformers v4 and v5. FLAN-T5 is **not** compatible with the causal `text-generation` pipeline. Input is capped at 512 tokens (longer prompts are truncated); production RAG systems need explicit context budgeting.

The optional demo is off by default. Set `RUN_PRETRAINED_MODEL = True` in Colab to try it. The mandatory notebook runs without internet, GPU, model downloads, or API keys.

In [ ]:
RUN_PRETRAINED_MODEL = False  # Set True in Colab to download and run a model

if RUN_PRETRAINED_MODEL:
    try:
        import importlib.util
        if (importlib.util.find_spec('transformers') is None or
                importlib.util.find_spec('sentencepiece') is None):
            %pip -q install transformers sentencepiece

        import torch
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

        # FLAN-T5 is an encoder-decoder model, so call .generate() directly.
        # This works with Transformers v4 and v5 (text2text-generation pipeline was removed in v5).
        model_name = 'google/flan-t5-small'
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = model.to(device)
        model.eval()

        def generate_answer(text):
            inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
            inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
            with torch.inference_mode():
                tokens = model.generate(**inputs, max_new_tokens=100, do_sample=False)
            return tokenizer.decode(tokens[0], skip_special_tokens=True)

        context = next(
            hit["text"] for hit in evidence
            if hit["report_id"] == "R02"
            and hit["text"].startswith("Action:")
        )

        question = "What action is recommended for gearbox B04?"

        prompt_simple = f"""
        Answer the question using the context.

        Context: {context}
        Question: {question}
        Answer:
        """

        print("Question:", question)
        print("LLM answer:", generate_answer(prompt_simple))
    except Exception as exc:
        print('Optional model demo unavailable:', type(exc).__name__, str(exc))
        print('The mandatory RAG exercise still works without a downloaded model.')
else:
    print('Optional transformer demo skipped. Set RUN_PRETRAINED_MODEL=True to activate.')

### Discussion / optional challenges

1. In your own words, contrast **GPT next-token prediction** with **BERT masked-token prediction** (Part 5).
2. Why does the same question fail when the required source paragraph is **not retrieved**?
3. How could you replace our lexical representation with **multilingual-E5 embeddings**? How is a query embedding compared with chunk embeddings?
4. In Part 6, why might a **reranker** be applied after a first retrieval stage?
5. What must you do before trusting a numerical answer extracted from an industrial report?
6. **Challenge:** deliberately delete report R02's action paragraph and see how the prompt changes. Can a correct system still know B04's recommended action?

**Takeaway:** Tokenization affects input length; a RAG system retrieves relevant evidence, builds a source-aware prompt, then optionally asks a language model to generate an answer. Retrieval relevance, answer generation, and factual verification are different stages.